# LeetCode #1144 — Decrease Elements To Make Array Zigzag

**Medium · Python 3 · Array · Greedy · Parity (chẵn/lẻ)**  
[Đề gốc](https://leetcode.com/problems/decrease-elements-to-make-array-zigzag/) · [Đối chiếu mô tả](https://leetcode.doocs.org/en/lc/1144/)

```python
class Solution:
    def movesToMakeZigzag(self, nums: List[int]) -> int:
        ...
```

**Câu hỏi xuyên suốt:** Chỉ được **giảm** phần tử (mỗi lần giảm 1), vậy làm thế nào để tính **ít thao tác nhất** mà không thử vô số cách thay đổi cả mảng?

## 1. Mục tiêu học tập

1. Hiểu hai dạng zigzag (`< > < >` hoặc `> < > <`) và điều kiện **nghiêm ngặt**.
2. Thành thạo index chẵn/lẻ qua toán tử `%`, kiểm tra hàng xóm `i-1`, `i+1`.
3. Tính số lần giảm **tối thiểu** để một vị trí thấp hơn hai bên.
4. Chứng minh vì sao chỉ cần xét **hai kiểu vị trí valley** và không giảm peak.
5. Viết lời giải greedy `O(n)` thời gian, `O(1)` bộ nhớ, không sửa input.
6. Trace, kiểm thử và thực hành 5 bài tập TODO.

## 2. Đề bài & 2 ví dụ chuẩn

Một **move** (thao tác) = chọn **một** phần tử và giảm **đúng 1**. Có thể thực hiện nhiều lần trên cùng phần tử.

Mảng **zigzag** nếu thỏa **một trong hai** dạng:

- **Dạng A:** `A[0] > A[1] < A[2] > A[3] < ...` (vị trí lẻ là **đáy / valley**).
- **Dạng B:** `A[0] < A[1] > A[2] < A[3] > ...` (vị trí chẵn là **đáy / valley**).

Trả **số move tối thiểu** để biến `nums` thành một mảng thỏa một trong hai dạng. Chỉ được **giảm**, không tăng.

| Input | Output | Giải thích |
|---|---:|---|
| `[1,2,3]` | **2** | Giảm 2 ở giữa thành 0, hoặc giảm 3 thành 1 |
| `[9,6,1,6,2]` | **4** | Chọn dạng có vị trí lẻ làm đáy: 6→0 tại index 1 cần 6? Không tối ưu; phương án tối ưu là vị trí chẵn làm đáy, tổng 4 |

**Constraints:** `1 <= len(nums) <= 1000`, `1 <= nums[i] <= 1000`. **Giá trị sau khi giảm có thể bằng 0 hoặc âm** — đề không cấm.

In [1]:
arr=[1,2,3]
print('Input:',arr)
print('Giảm vị trí giữa 2 lần:', [1,0,3])
print('Giảm phần tử cuối 2 lần:', [1,2,1])
assert 1>0<3 and 1<2>1

Input: [1, 2, 3]
Giảm vị trí giữa 2 lần: [1, 0, 3]
Giảm phần tử cuối 2 lần: [1, 2, 1]


## 3. Python căn bản: index chẵn / lẻ

Các vị trí tính từ **0**:

- `i % 2 == 0` ⇔ `i` **chẵn** (0,2,4,...).
- `i % 2 == 1` ⇔ `i` **lẻ** (1,3,5,...).
- `range(parity, n, 2)` duyệt mọi index có cùng parity với `parity`.

Trong bài này, `parity=0` nghĩa là **các vị trí chẵn cần làm đáy**, `parity=1` nghĩa là **các vị trí lẻ cần làm đáy**. Đừng nhầm parity với **giá trị** chẵn/lẻ của `nums[i]`.

In [2]:
nums_demo=[9,6,1,6,2]
print('Index chẵn (đáy dạng B):',list(range(0,len(nums_demo),2)))
print('Index lẻ (đáy dạng A):',list(range(1,len(nums_demo),2)))
for i,x in enumerate(nums_demo):
    print('i=',i,'x=',x,'parity=',i%2)

Index chẵn (đáy dạng B): [0, 2, 4]
Index lẻ (đáy dạng A): [1, 3]
i= 0 x= 9 parity= 0
i= 1 x= 6 parity= 1
i= 2 x= 1 parity= 0
i= 3 x= 6 parity= 1
i= 4 x= 2 parity= 0


## 4. «Đáy» và «đỉnh»: bất đẳng thức nghiêm ngặt

Muốn `nums[i]` làm **đáy**, phải có `nums[i] < nums[i-1]` (nếu có bên trái) và `nums[i] < nums[i+1]` (nếu có bên phải).

Vì các phần tử là số nguyên, nếu hàng xóm nhỏ nhất bằng `k` thì đáy **lớn nhất được phép** là `k-1`, không phải `k`. Ví dụ hai hàng xóm là `3` và `5`, đáy phải `<=2`.

**Dấu `<` khác `<=`**: hai phần tử bằng nhau *không tạo zigzag hợp lệ*.

In [3]:
left,right=3,5
allowed=min(left,right)-1
print('Hai hàng xóm:',left,right,'-> đáy cao nhất hợp lệ:',allowed)
assert allowed==2 and 2<3 and 2<5

Hai hàng xóm: 3 5 -> đáy cao nhất hợp lệ: 2


## 5. Công thức tính số bước giảm tại một vị trí

Với vị trí **đáy** `i`, gọi `neighbor_min` là giá trị nhỏ nhất giữa các hàng xóm **tồn tại**.

Đáy phải không vượt `neighbor_min-1`. Nếu `nums[i]` đã đủ nhỏ, giảm `0` bước. Nếu chưa:

\[moves_i = \max\big(0,\ nums[i] - neighbor\_min + 1\big).\]

Ví dụ `nums=[9,6,1]`, tại `i=1`: `neighbor_min=min(9,1)=1` ⇒ phải giảm `6` xuống `0`, tốn `6` bước.

In [4]:
center=6
neighbor_min=min(9,1)
need=max(0,center-neighbor_min+1)
print('Cần giảm:',need,'| Giá trị sau giảm:',center-need)
assert need==6 and center-need==0

Cần giảm: 6 | Giá trị sau giảm: 0


## 6. Ranh giới mảng: đầu và cuối chỉ có một hàng xóm

- `i=0`: chỉ có hàng xóm bên phải (nếu `n>1`).
- `i=n-1`: chỉ có hàng xóm bên trái (nếu `n>1`).
- `n=1`: không có hàng xóm, **cả hai dạng zigzag đều hợp lệ**, đáp án 0.

Ta có thể dùng `float('inf')` cho bên không tồn tại. Vì mọi số hữu hạn đều nhỏ hơn vô cực, phép `min` tự chọn hàng xóm **thực sự tồn tại**. Đặc biệt với `n=1`, `min(inf,inf)=inf` nên `max(0, num-inf+1)=0`.

In [5]:
INF=float('inf')
for a in ([7],[1,2],[2,1]):
    at0=min(INF, a[1] if len(a)>1 else INF)
    moves=max(0,a[0]-at0+1)
    print(a,'| để index 0 làm đáy cần',moves,'lần giảm')
assert max(0,7-INF+1)==0

[7] | để index 0 làm đáy cần 0 lần giảm
[1, 2] | để index 0 làm đáy cần 0 lần giảm
[2, 1] | để index 0 làm đáy cần 2 lần giảm


## 7. Vì sao không cần giảm các «đỉnh»?

Ta đang tìm số bước **ít nhất** và chỉ được **giảm** giá trị. Nếu một vị trí phải là **đỉnh** (`>` hai bên), việc giảm đỉnh không thể làm nó cao hơn hàng xóm; trái lại chỉ khiến điều kiện đỉnh **khó đạt hơn**.

Do đó, khi đã chọn kiểu zigzag, ta luôn có một phương án tối ưu **không giảm bất kỳ đỉnh nào**; chỉ sửa các đáy khi cần. Đây là bước đột phá Greedy, thay vì thử biến đổi mọi phần tử.

## 8. Vì sao các đáy có thể xử lý **độc lập**?

Các vị trí đáy có cùng parity cách nhau **2 chỉ số**, không bao giờ nằm cạnh nhau. Mỗi đáy chỉ cần thấp hơn các **đỉnh** liền kề, mà những đỉnh đó ta giữ nguyên.

Bởi vậy số move cho một đáy **không ảnh hưởng** số move cho đáy khác. Cộng số move tối thiểu từng đáy sẽ cho tổng tối thiểu cho **dạng zigzag đã chọn**.

Cuối cùng, vì đề chấp nhận **một trong hai dạng**, ta lấy `min(cost_even_valleys, cost_odd_valleys)`.

## 9. Cách trực giác trước tối ưu: thử giảm từng giá trị (chỉ mảng nhỏ)

Nếu một mảng rất ngắn có các giá trị nhỏ, ta có thể thử **mọi giá trị cuối** không lớn hơn ban đầu trong một khoảng hữu hạn và chọn mảng zigzag có tổng số bước nhỏ nhất. Cách này vừa chậm vừa không phù hợp giới hạn `n=1000`.

Ta chỉ dùng nó như **oracle kiểm thử cho mảng dài tối đa 5, giá trị nhỏ**. Phương án chính dưới đây là `O(n)`.

In [6]:
from itertools import product

def is_zigzag(a):
    n=len(a)
    return any(all(a[i] < (a[i-1] if i else float('inf'))
                       and a[i] < (a[i+1] if i+1<n else float('inf'))
                       for i in range(parity,n,2))
               for parity in (0,1))

def zigzag_bruteforce(nums):
    """Oracle giới hạn: độ dài <= 5, giá trị nhỏ; cho phép giảm xuống tối thiểu -1."""
    best=float('inf')
    # Với input dương nhỏ, -1 là đủ cho một phương án tối ưu:
    # chỉ cần thấp hơn tối thiểu 1 đơn vị so với các đỉnh >=1.
    ranges=[range(-1,x+1) for x in nums]
    for cand in product(*ranges):
        cost=sum(x-y for x,y in zip(nums,cand))
        if cost>=best: continue
        if is_zigzag(cand): best=cost
    return best

print('Vét cạn [1,2,3]:',zigzag_bruteforce([1,2,3]))
assert zigzag_bruteforce([1,2,3])==2

Vét cạn [1,2,3]: 2


## 10. Greedy: tính chi phí một kiểu đáy

Hàm `cost_for_valleys(nums, parity)` sẽ đi qua các index phải là đáy (`0,2,4,...` hoặc `1,3,5,...`).

1. Tìm `left`/`right` (giá trị `inf` nếu vượt biên).
2. Tính `need=max(0, nums[i]-min(left,right)+1)`.
3. Cộng `need` vào `total`.

Lưu ý **không sửa `nums`**: mọi đỉnh được giữ nguyên; các đáy độc lập, chỉ cần tính cost. Điều này giúp tránh lỗi hiệu ứng phụ giữa hai phương án.

In [7]:
def cost_for_valleys(nums, parity):
    n=len(nums)
    total=0
    for i in range(parity,n,2):
        left=nums[i-1] if i>0 else float('inf')
        right=nums[i+1] if i+1<n else float('inf')
        need=max(0,nums[i]-min(left,right)+1)
        total+=need
    return total

for a in ([1,2,3],[9,6,1,6,2],[1],[2,2,2]):
    even=cost_for_valleys(a,0)
    odd=cost_for_valleys(a,1)
    print(a,'| index chẵn đáy:',even,'| index lẻ đáy:',odd,'| tối ưu:',min(even,odd))
assert min(cost_for_valleys([9,6,1,6,2],0), cost_for_valleys([9,6,1,6,2],1))==4

[1, 2, 3] | index chẵn đáy: 2 | index lẻ đáy: 2 | tối ưu: 2
[9, 6, 1, 6, 2] | index chẵn đáy: 4 | index lẻ đáy: 12 | tối ưu: 4
[1] | index chẵn đáy: 0 | index lẻ đáy: 0 | tối ưu: 0
[2, 2, 2] | index chẵn đáy: 2 | index lẻ đáy: 1 | tối ưu: 1


## 11. Trace bằng tay: `[9,6,1,6,2]`

**Cho vị trí chẵn (0,2,4) làm đáy:**

| i | nums[i] | Hàng xóm | Muốn đáy ≤ | Số move |
|---:|---:|---|---:|---:|
| 0 | 9 | phải=6 | 5 | 4 |
| 2 | 1 | trái=6, phải=6 | 5 | 0 |
| 4 | 2 | trái=6 | 5 | 0 |

Tổng **4**. Mảng mới có thể là `[5,6,1,6,2]`.

**Cho vị trí lẻ (1,3) làm đáy:** `i=1` cần `6` bước để thấp hơn `1`, `i=3` cần `6` bước để thấp hơn `1` ⇒ tổng **12**.

Chọn **min(4,12)=4**. Mẹo: không nhất thiết mỗi vị trí đáy đều phải giảm; nếu đã nhỏ hơn hai hàng xóm, cost là 0.

In [8]:
def trace_valleys(nums,parity):
    total=0
    print('nums=',nums,'| valley parity=',parity)
    for i in range(parity,len(nums),2):
        left=nums[i-1] if i>0 else float('inf')
        right=nums[i+1] if i+1<len(nums) else float('inf')
        limit=min(left,right)-1
        need=max(0,nums[i]-limit)
        total+=need
        print(f'i={i}, gốc={nums[i]}, trái={left}, phải={right}, '
              f'max đáy={limit}, giảm={need}, tổng={total}')
    return total

assert trace_valleys([9,6,1,6,2],0)==4
assert trace_valleys([9,6,1,6,2],1)==12

nums= [9, 6, 1, 6, 2] | valley parity= 0
i=0, gốc=9, trái=inf, phải=6, max đáy=5, giảm=4, tổng=4
i=2, gốc=1, trái=6, phải=6, max đáy=5, giảm=0, tổng=4
i=4, gốc=2, trái=6, phải=inf, max đáy=5, giảm=0, tổng=4
nums= [9, 6, 1, 6, 2] | valley parity= 1
i=1, gốc=6, trái=9, phải=1, max đáy=0, giảm=6, tổng=6
i=3, gốc=6, trái=1, phải=2, max đáy=0, giảm=6, tổng=12


## 12. Code giải chính: so đúng hai phương án

Vì mảng zigzag chỉ có **hai kiểu chẵn/lẻ** (xét vị trí đáy), chỉ cần tính cả hai rồi lấy nhỏ nhất.

Trong code submit, ta có thể gộp vòng lặp vào một hàm bên trong `Solution`, hoặc dùng hai biến `cost[0]`, `cost[1]`. Bản dưới trả nguyên đúng signature của LeetCode.

In [9]:
from typing import List

class Solution:
    def movesToMakeZigzag(self, nums: List[int]) -> int:
        n=len(nums)
        cost=[0,0]
        for parity in (0,1):
            for i in range(parity,n,2):
                left=nums[i-1] if i>0 else float('inf')
                right=nums[i+1] if i+1<n else float('inf')
                need=max(0,nums[i]-min(left,right)+1)
                cost[parity]+=need
        return min(cost)

sol=Solution()
assert sol.movesToMakeZigzag([1,2,3])==2
assert sol.movesToMakeZigzag([9,6,1,6,2])==4
print('LeetCode Solution: PASS')

LeetCode Solution: PASS


## 13. Có thể dùng `i%2` để tính cả hai cost trong một lượt

Ở mỗi index `i`, ta có một chi phí để **biến chính index ấy thành đáy**; cộng nó vào `cost[i%2]`. Cách này tương đương với hai vòng `parity` trước, nhưng code ngắn hơn.

Với người mới, bản hai vòng lặp dễ nhìn hơn vì thể hiện rõ hai trường hợp. Hai bản đều chạy `O(n)`.

In [10]:
def zigzag_one_pass(nums):
    cost=[0,0]
    n=len(nums)
    for i,x in enumerate(nums):
        left=nums[i-1] if i>0 else float('inf')
        right=nums[i+1] if i+1<n else float('inf')
        cost[i%2]+=max(0,x-min(left,right)+1)
    return min(cost)

for arr in ([1,2,3],[9,6,1,6,2],[1],[2,2,2]):
    assert zigzag_one_pass(arr)==Solution().movesToMakeZigzag(arr)
print('Một lượt == hai lượt: PASS')

Một lượt == hai lượt: PASS


## 14. Vì sao Greedy đúng? Chứng minh không bỏ sót

1. Mọi mảng zigzag hợp lệ phải thuộc **một trong hai kiểu** đã xét.
2. Khi cố định kiểu, ta chỉ cần giảm các **đáy**. Nếu một phương án có giảm đỉnh, hoàn trả đỉnh về giá trị ban đầu (cao hơn) không làm vi phạm điều kiện; số move không tăng. Vậy tồn tại phương án tối ưu không giảm đỉnh.
3. Các đáy không kề nhau; hàng xóm của đáy luôn là đỉnh **giữ nguyên**. Vì thế điều kiện từng đáy độc lập. Cách giảm đáy đến **giá trị lớn nhất vẫn nhỏ hơn hàng xóm** chính là cách ít move nhất tại vị trí ấy.
4. Cộng chi phí từng đáy là tối ưu cho kiểu đó; lấy min của hai kiểu là tối ưu toàn bài.

## 15. Phân tích Time & Space Complexity

Gọi `n=len(nums)`.

- Mỗi index thuộc đúng một parity. Dù tính hai vòng `parity`, tổng số index xử lý là `n` ⇒ **O(n)** thời gian.
- Chỉ lưu `n`, `cost=[0,0]`, `left`, `right`, `need` ⇒ **O(1)** bộ nhớ phụ.
- **Không** sửa `nums`, nên gọi lại hàm cùng input sẽ cho cùng output.

Ngược lại, brute force thử nhiều dãy giá trị đã giảm nên số trường hợp tăng theo tích số lựa chọn tại từng vị trí — không phù hợp `n=1000`.

## 16. Test mẫu, biên, mảng bằng nhau, đã zigzag

Trường hợp `n=1` phải trả `0`. Mảng `n=2` cần đúng **một bất đẳng thức**; một trong hai kiểu thường hợp lệ ngay nếu hai số khác nhau. Với số bằng nhau cần ít nhất một bước.

Quan trọng: không bắt giá trị **sau giảm** phải dương, dù input ban đầu dương. Ví dụ `[1,1]` cần 1 move để thành `[0,1]` hoặc `[1,0]`.

In [11]:
cases=[
    ([1],0),([1000],0),([1,2],0),([2,1],0),
    ([1,1],1),([1,2,3],2),([9,6,1,6,2],4),
    ([1,3,1,3,1],0),([3,1,3,1,3],0),([2,2,2],1),
]
for arr,expected in cases:
    original=arr.copy()
    actual=Solution().movesToMakeZigzag(arr)
    assert actual==expected,(arr,actual,expected)
    assert arr==original,'Hàm không được sửa mảng input'
    print('PASS',arr,'->',actual)

PASS [1] -> 0
PASS [1000] -> 0
PASS [1, 2] -> 0
PASS [2, 1] -> 0
PASS [1, 1] -> 1
PASS [1, 2, 3] -> 2
PASS [9, 6, 1, 6, 2] -> 4
PASS [1, 3, 1, 3, 1] -> 0
PASS [3, 1, 3, 1, 3] -> 0
PASS [2, 2, 2] -> 1


## 17. Test chéo ngẫu nhiên với brute force độc lập

Brute force ở mục 9 thử mọi mảng sau giảm trong phạm vi đủ nhỏ. Đây là cách kiểm tra tính đúng của **lập luận Greedy**, chứ không chỉ xác nhận mã chạy không lỗi.

Ta chỉ sinh **mảng rất ngắn, giá trị 1..3**, để tổng số tổ hợp không quá lớn. Không dùng oracle này với input 1000 phần tử.

In [12]:
import random
rng=random.Random(1144)
for _ in range(170):
    arr=[rng.randint(1,3) for _ in range(rng.randint(1,5))]
    expected=zigzag_bruteforce(arr)
    actual=Solution().movesToMakeZigzag(arr)
    assert expected==actual,(arr,expected,actual)
    assert zigzag_one_pass(arr)==actual
print('PASS 170 ca đối chiếu với brute force')

PASS 170 ca đối chiếu với brute force


## 18. Test giới hạn `n=1000`, giá trị tối đa 1000

Trong một input dài toàn phần tử bằng nhau, cần giảm các đáy cùng parity một bước mỗi vị trí; **lấy kiểu có ít đáy hơn**. Với `n=1000`, hai kiểu đều có đúng 500 vị trí ⇒ 500 bước.

In [13]:
big=[1000]*1000
assert Solution().movesToMakeZigzag(big)==500
assert zigzag_one_pass(big)==500
assert big==[1000]*1000
print('PASS n=1000, mọi giá trị 1000 -> 500 bước')

PASS n=1000, mọi giá trị 1000 -> 500 bước


## 19. Những lỗi rất dễ mắc

| Sai | Vì sao | Sửa |
|---|---|---|
| Dùng `<=` thay cho `<` | Bằng nhau **không** zigzag | Thêm `+1` trong cost |
| Chỉ kiểm tra một bên | Vị trí giữa phải thấp hơn **cả hai** | `min(left,right)` |
| Chỉ xem một dạng zigzag | Có thể bỏ đáp án tốt hơn | Thử parity `0` và `1` |
| Giảm trực tiếp `nums` khi thử một dạng | Làm biến đổi input cho dạng sau | Chỉ tính **chi phí**, không sửa input |
| Nhầm parity index và parity giá trị | Ở đây là chẵn/lẻ **vị trí** | Dùng `i%2` |
| Cố giữ các số sau giảm dương | Đề chỉ giới hạn **input** dương | Cho phép 0 hoặc âm |
| Trả mảng mới | Sai đầu ra | Trả **số move tối thiểu** |

## 20. Pattern: thử số ít «dạng cấu hình», rồi greedy từng vị trí

Nếu đề có một cấu trúc **luân phiên theo parity**, thường chỉ có vài dạng toàn cục cần thử (`even`/`odd`). Sau khi cố định dạng, hỏi: «Mỗi vị trí còn phụ thuộc vào vị trí nào? Có thể tính riêng không?».

Bài này đặc biệt thuận lợi vì chỉ được giảm và các đáy **không kề nhau**. Nếu được **cả tăng lẫn giảm** mà chi phí khác nhau, hoặc các vị trí cần sửa **kề nhau**, lập luận độc lập hiện tại có thể không còn đúng.

## 21. Năm bài tập từ dễ đến nâng cao

1. **Dự đoán:** `[2,2]` cần tối thiểu bao nhiêu move? Kết quả có thể là những mảng nào?
2. **Tính tay:** Với `[3,1,3,1]`, chi phí của parity chẵn/lẻ lần lượt là bao nhiêu?
3. **TODO:** `exercise_cost(nums, parity)` tính chi phí khi parity ấy làm đáy.
4. **TODO:** `exercise_zigzag(nums)` gọi hàm bài 3 cho hai dạng, trả min; không sửa `nums`.
5. **Mini challenge:** viết `exercise_construct(nums)` trả `(số_move, mảng_zigzag)`. Nếu hai kiểu cùng tốt, ưu tiên parity `0`. Kiểm tra tổng mức giảm và tính zigzag của mảng trả về.

In [14]:
# BÀI TẬP TODO — chưa chạy, để bạn tự viết.
def exercise_cost(nums, parity):
    # TODO: duyệt index parity, parity+2,...; tính left/right; cộng số move.
    pass

def exercise_zigzag(nums):
    # TODO: tính cost cho parity 0 và 1 rồi lấy min.
    pass

def exercise_construct(nums):
    # TODO: chọn parity tối ưu, tạo copy của nums;
    # giảm các đáy trong bản sao, trả (cost, bản sao).
    pass

# Khi làm xong, bỏ # để test:
# assert exercise_cost([9,6,1,6,2],0)==4
# assert exercise_zigzag([1,2,3])==2
# c,b = exercise_construct([9,6,1,6,2]); assert c==4 and is_zigzag(b)

## 22. Gợi ý 3 mức — thử suy nghĩ trước

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Hai số bằng nhau | Phải có một dấu `<` hay `>` | 1 move ⇒ `[1,2]` hoặc `[2,1]` |
| 2 | Vị trí lẻ đã nhỏ hơn hàng xóm? | Kiểm tra hai đỉnh ở index chẵn | Cost chẵn=6, cost lẻ=0 |
| 3 | Mỗi đáy độc lập | Hàng xóm ngoài mảng = inf | `max(0,nums[i]-min(left,right)+1)` |
| 4 | Hai dạng duy nhất | `range(2)` | `min(exercise_cost(nums,0),exercise_cost(nums,1))` |
| 5 | Không sửa input | `copy()` | `new[i] -= need` tại các đáy |

## 23. Đáp án tham khảo — chỉ đọc sau khi tự giải

**Bài 1:** `1` move; có thể `[1,2]` hoặc `[2,1]`.  
**Bài 2:** Với `[3,1,3,1]`: parity **chẵn làm đáy** phải giảm 3→0 ở hai index chẵn (mỗi chỗ 3 bước) ⇒ **6**; parity **lẻ làm đáy** đã đúng ⇒ **0**.

Chú ý bài 2: tại mỗi index chẵn, `3 - 1 + 1 = 3` bước giảm. Có hai index chẵn nên tổng `3+3=6`.

In [15]:
def answer_cost(nums,parity):
    total=0
    for i in range(parity,len(nums),2):
        left=nums[i-1] if i else float('inf')
        right=nums[i+1] if i+1<len(nums) else float('inf')
        total+=max(0,nums[i]-min(left,right)+1)
    return total

def answer_zigzag(nums):
    return min(answer_cost(nums,0),answer_cost(nums,1))

def answer_construct(nums):
    c0,c1=answer_cost(nums,0),answer_cost(nums,1)
    parity=0 if c0<=c1 else 1
    result=nums.copy()
    for i in range(parity,len(nums),2):
        left=nums[i-1] if i else float('inf')
        right=nums[i+1] if i+1<len(nums) else float('inf')
        result[i] -= max(0,nums[i]-min(left,right)+1)
    return min(c0,c1),result

assert answer_cost([9,6,1,6,2],0)==4
assert answer_cost([3,1,3,1],0)==6
assert answer_cost([3,1,3,1],1)==0
assert answer_zigzag([1,2,3])==2
for arr in ([1],[1,2,3],[9,6,1,6,2],[2,2,2]):
    c,b=answer_construct(arr)
    assert c==Solution().movesToMakeZigzag(arr)
    assert is_zigzag(b)
    assert all(y<=x for x,y in zip(arr,b))
    assert sum(x-y for x,y in zip(arr,b))==c
print('Đáp án bài tập: PASS')

Đáp án bài tập: PASS


## 24. Tổng kết & checklist submit

```text
Đáp án = min(cost(parity=0), cost(parity=1))
Với i là vị trí đáy:
  left  = nums[i-1] hoặc inf nếu không tồn tại
  right = nums[i+1] hoặc inf nếu không tồn tại
  moves = max(0, nums[i] - min(left, right) + 1)
  total += moves
Không thay đổi các vị trí đỉnh; không sửa input.
```

- [ ] Dùng **vị trí chẵn/lẻ**, không phải giá trị chẵn/lẻ.
- [ ] So sánh nghiêm ngặt `<`, nhớ `+1`.
- [ ] Xử lý `n=1`, hai đầu, các phần tử bằng nhau.
- [ ] Cả hai dạng zigzag đều được xét.
- [ ] Code nộp trả **int**, đạt `O(n)` thời gian / `O(1)` bộ nhớ phụ.